In [0]:
import os, sys
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.append("/Workspace" + os.path.dirname(notebook_path))

from config_prod import window_days, config_params
from si2_functions import apply_bayesian_smoothing, apply_threshold_and_cap, calculate_penalties, generate_seller_index

In [0]:
# --- MAIN EXECUTION ---
m_metrics = spark.table("marketplace_olist.gold.fact_seller_metrics")

# Using lambda to pass the params dict to the transform functions
gold_seller_index = (
    m_metrics
    .transform(lambda df: apply_bayesian_smoothing(df, config_params, window_days))
    .transform(lambda df: apply_threshold_and_cap(df, config_params))
    .transform(lambda df: calculate_penalties(df, config_params))
    .transform(generate_seller_index)
)

In [0]:
# Save to Gold Medallion Layer
gold_seller_index.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("marketplace_olist.gold.fact_seller_index")